In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.0 MB/s eta 0:00:00


In [2]:
import os
import json
import torch
import numpy as np
import faiss
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
import torch.nn.functional as F
import shutil

# ─────────────────────────────────────────
# 1. GPU Check
# ─────────────────────────────────────────
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — "
          f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

DEVICE = "cuda:0"

# ─────────────────────────────────────────
# 2. Paths
# ─────────────────────────────────────────
raw_dataset_dir  = '/kaggle/input/datasets/hades199/vr-final-project-dataset/vr-final-project-dataset'
partition_file   = os.path.join(raw_dataset_dir, 'Eval/list_eval_partition.txt')
bbox_file        = os.path.join(raw_dataset_dir, 'Anno/list_bbox_inshop.txt') 
cropped_dir      = '/kaggle/input/datasets/hades199/3c-yolo-cropped-images'
caption_file     = '/kaggle/input/datasets/hades199/3c-blip-captions/captions.json'
output_dir       = '/kaggle/working/faiss_indexes'
os.makedirs(output_dir, exist_ok=True)

# Fine-tuned model seeds — same as CLIP fine-tuning script
SEEDS               = [42, 601, 606, 619, 623]
clip_finetuned_base = '/kaggle/input/datasets/hades199/3c-clip-fintuned-model/clip_finetuned_v2'

# Two alpha values for Ablations B and C (per spec: any two values in [0,1])
ALPHA_VALUES = [0.5, 0.7]

# ─────────────────────────────────────────
# 3. Download base CLIP ONCE then go offline
#    Prevents HuggingFace network hanging
# ─────────────────────────────────────────
BASE_CLIP_CACHE = '/kaggle/working/clip_base_pretrained'

# Allow download
os.environ["TRANSFORMERS_OFFLINE"] = "0"

if not os.path.exists(BASE_CLIP_CACHE):
    print("\nDownloading base pretrained CLIP once...")
    base_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    base_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    base_model.save_pretrained(BASE_CLIP_CACHE)
    base_proc.save_pretrained(BASE_CLIP_CACHE)
    del base_model
    torch.cuda.empty_cache()
    print(f"✅ Base CLIP cached at {BASE_CLIP_CACHE}")
else:
    print(f"\n✅ Base CLIP already cached at {BASE_CLIP_CACHE}")

# Force offline mode — NO more HuggingFace calls after this point
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("✅ Transformers set to offline mode — no network calls from here.")

# ─────────────────────────────────────────
# 4. Verify all fine-tuned seed paths exist
# ─────────────────────────────────────────
print("\nVerifying fine-tuned CLIP seed paths...")
for seed in SEEDS:
    seed_path = os.path.join(clip_finetuned_base, f'seed_{seed}', 'best')
    status    = "✅" if os.path.exists(seed_path) else "❌ NOT FOUND"
    print(f"  seed_{seed}: {status}")

# ─────────────────────────────────────────
# 5. Load Partition + Bbox — Gallery split only
#    FIX: merge bbox to get clothes_type
# ─────────────────────────────────────────
print("\nLoading partition metadata...")
df_part    = pd.read_csv(partition_file, sep=r'\s+', skiprows=1)
df_bbox    = pd.read_csv(bbox_file,      sep=r'\s+', skiprows=1)

# Merge to get clothes_type (1: Upper, 2: Lower, 3: Full)
df_part = pd.merge(df_part, df_bbox[['image_name', 'clothes_type']], on='image_name', how='left')

df_gallery = df_part[df_part['evaluation_status'] == 'gallery'].reset_index(drop=True)

def extract_item_id(image_name):
    parts = image_name.replace('\\', '/').split('/')
    for part in parts:
        if part.startswith('id_'):
            return part
    return None

df_gallery['item_id'] = df_gallery['image_name'].apply(extract_item_id)
df_gallery = df_gallery.dropna(subset=['item_id']).reset_index(drop=True)

# Map clothes_type to human-readable names
CLASS_MAP = {1: 'Upper', 2: 'Lower', 3: 'Full'}
df_gallery['class_name'] = df_gallery['clothes_type'].map(CLASS_MAP).fillna('Unknown')

print(f"Gallery images  : {len(df_gallery)}")
print(f"Unique item IDs : {df_gallery['item_id'].nunique()}")
print(f"Class distribution:")
print(f"  Upper: {(df_gallery['class_name']=='Upper').sum()}")
print(f"  Lower: {(df_gallery['class_name']=='Lower').sum()}")
print(f"  Full : {(df_gallery['class_name']=='Full').sum()}")

# ─────────────────────────────────────────
# 6. Load Captions
# ─────────────────────────────────────────
with open(caption_file, 'r') as f:
    captions = json.load(f)
print(f"Captions loaded : {len(captions)}")

# ─────────────────────────────────────────
# 7. Helper — resolve cropped image path
# ─────────────────────────────────────────
def resolve_crop_path(image_name):
    p1 = os.path.join(cropped_dir, image_name)
    p2 = os.path.join(cropped_dir, image_name.replace('/', '_'))
    if os.path.exists(p1): return p1
    if os.path.exists(p2): return p2
    return None

# ─────────────────────────────────────────
# 8. Embedding Function
# ─────────────────────────────────────────
EMBED_DIM  = 512
BATCH_SIZE = 128

def get_fused_embeddings(clip_model, processor, images, texts, alpha, device):
    """
    vi = α · ϕV(image) + (1−α) · ϕT(caption)
    ‖vi‖ = 1  (as per project spec)
    """
    inputs = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77
    ).to(device)

    with torch.no_grad():
        # ✅ Go through vision_model explicitly to always get a clean tensor
        vision_outputs = clip_model.vision_model(
            pixel_values=inputs["pixel_values"]
        )
        image_embeds = clip_model.visual_projection(
            vision_outputs.pooler_output
        )

        if alpha < 1.0:
            text_outputs = clip_model.text_model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"]
            )
            text_embeds = clip_model.text_projection(
                text_outputs.pooler_output
            )
        else:
            # Ablation A: pure vision — text not used
            text_embeds = torch.zeros_like(image_embeds)

    image_embeds = F.normalize(image_embeds, dim=-1)
    text_embeds  = F.normalize(text_embeds,  dim=-1)
    fused        = alpha * image_embeds + (1 - alpha) * text_embeds
    fused        = F.normalize(fused, dim=-1)

    return fused.cpu().numpy().astype(np.float32)

# ─────────────────────────────────────────
# 9. Core Indexing Function
#    FIX: stores clothes_type in metadata
# ─────────────────────────────────────────
def build_index(clip_model, processor, alpha, save_name):
    """Build and save one FAISS HNSW index."""
    hnsw = faiss.IndexHNSWFlat(EMBED_DIM, 32)
    hnsw.hnsw.efConstruction = 200
    index = faiss.IndexIDMap(hnsw)

    metadata = {}
    faiss_id = 0
    skipped  = 0

    batch_images, batch_texts, batch_rows = [], [], []

    for _, row in tqdm(df_gallery.iterrows(),
                       total=len(df_gallery),
                       desc=f"  [{save_name}]"):

        image_name  = row['image_name']
        item_id     = row['item_id']
        class_name  = row['class_name']        # ← ADDED
        crop_path   = resolve_crop_path(image_name)

        if not crop_path:
            skipped += 1
            continue

        try:
            img = Image.open(crop_path).convert("RGB")
        except Exception:
            skipped += 1
            continue

        rel_key = os.path.relpath(crop_path, cropped_dir)
        caption = captions.get(rel_key,
                  captions.get(image_name.replace('/', '_'),
                  "a clothing item"))

        batch_images.append(img)
        batch_texts.append(caption)
        batch_rows.append({
            "image_name":   image_name,
            "item_id":      item_id,
            "clothes_type": class_name,          # ← ADDED
        })

        if len(batch_images) == BATCH_SIZE:
            embeddings = get_fused_embeddings(
                clip_model, processor,
                batch_images, batch_texts, alpha, DEVICE
            )
            ids = np.arange(faiss_id, faiss_id + len(embeddings), dtype=np.int64)
            index.add_with_ids(embeddings, ids)
            for j, meta in enumerate(batch_rows):
                metadata[int(faiss_id + j)] = meta
            faiss_id += len(embeddings)
            batch_images, batch_texts, batch_rows = [], [], []

    # Last partial batch
    if batch_images:
        embeddings = get_fused_embeddings(
            clip_model, processor,
            batch_images, batch_texts, alpha, DEVICE
        )
        ids = np.arange(faiss_id, faiss_id + len(embeddings), dtype=np.int64)
        index.add_with_ids(embeddings, ids)
        for j, meta in enumerate(batch_rows):
            metadata[int(faiss_id + j)] = meta
        faiss_id += len(embeddings)

    # Set efSearch before saving
    hnsw.hnsw.efSearch = 128

    # Save index + metadata
    index_path    = os.path.join(output_dir, f"{save_name}_index.bin")
    metadata_path = os.path.join(output_dir, f"{save_name}_metadata.json")
    faiss.write_index(index, index_path)
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f)

    print(f"  ✅ Indexed: {faiss_id} | Skipped: {skipped} | Saved: {save_name}")

# ─────────────────────────────────────────
# 10. ABLATION A — Vision-only, α=1.0
#     1 index | frozen pretrained CLIP
# ─────────────────────────────────────────
print(f"\n{'='*55}")
print("ABLATION A — Vision-only CLIP (α=1.0, frozen)")
print(f"{'='*55}")

clip_model = CLIPModel.from_pretrained(BASE_CLIP_CACHE).to(DEVICE)
processor  = CLIPProcessor.from_pretrained(BASE_CLIP_CACHE)
clip_model.eval()

build_index(clip_model, processor,
            alpha=1.0,
            save_name="A_vision_only_alpha1.0")

del clip_model
torch.cuda.empty_cache()

# ─────────────────────────────────────────
# 11. ABLATION B — Frozen CLIP + BLIP-2 captions
#     2 indexes (one per alpha value)
# ─────────────────────────────────────────
print(f"\n{'='*55}")
print(f"ABLATION B — Frozen CLIP + BLIP-2 (α={ALPHA_VALUES})")
print(f"{'='*55}")

clip_model = CLIPModel.from_pretrained(BASE_CLIP_CACHE).to(DEVICE)
processor  = CLIPProcessor.from_pretrained(BASE_CLIP_CACHE)
clip_model.eval()

for alpha in ALPHA_VALUES:
    print(f"\n  Building B with α={alpha}...")
    build_index(clip_model, processor,
                alpha=alpha,
                save_name=f"B_frozen_alpha{alpha}")

del clip_model
torch.cuda.empty_cache()

# ─────────────────────────────────────────
# 12. ABLATION C — Fine-tuned CLIP + BLIP-2
#     5 seeds × 2 alpha = 10 indexes
#     eval script will compute mean ± std across seeds
# ─────────────────────────────────────────
print(f"\n{'='*55}")
print(f"ABLATION C — Fine-tuned CLIP + BLIP-2")
print(f"Seeds: {SEEDS} | α values: {ALPHA_VALUES}")
print(f"{'='*55}")

for seed in SEEDS:
    seed_model_path = os.path.join(clip_finetuned_base, f'seed_{seed}', 'best')

    if not os.path.exists(seed_model_path):
        print(f"\n  ⚠️  Skipping seed {seed} — path not found: {seed_model_path}")
        continue

    print(f"\n  Loading fine-tuned CLIP — seed {seed}...")
    clip_model = CLIPModel.from_pretrained(seed_model_path).to(DEVICE)
    processor  = CLIPProcessor.from_pretrained(seed_model_path)
    clip_model.eval()

    for alpha in ALPHA_VALUES:
        print(f"\n  Building C — seed={seed}, α={alpha}...")
        build_index(clip_model, processor,
                    alpha=alpha,
                    save_name=f"C_finetuned_seed{seed}_alpha{alpha}")

    del clip_model
    torch.cuda.empty_cache()

# ─────────────────────────────────────────
# 13. Summary
# ─────────────────────────────────────────
print(f"\n{'='*55}")
print("All indexes built!")
print(f"{'='*55}")
all_files  = sorted(os.listdir(output_dir))
total_size = 0
for fname in all_files:
    fpath = os.path.join(output_dir, fname)
    size  = os.path.getsize(fpath) / 1e6
    total_size += size
    print(f"  {fname}  ({size:.1f} MB)")
print(f"\n  Total size: {total_size:.1f} MB")

# ─────────────────────────────────────────
# 14. Zip for download
# ─────────────────────────────────────────
print(f"\nZipping FAISS indexes...")
zip_path = shutil.make_archive(
    '/kaggle/working/faiss_indexes',
    'zip',
    '/kaggle/working',
    'faiss_indexes'
)
print(f"✅ Zip saved to: {zip_path}")
print("\nNext step → Evaluation script (Recall@K, NDCG@K, mAP@K)")


GPUs available: 2
  GPU 0: Tesla T4 — 15.6 GB
  GPU 1: Tesla T4 — 15.6 GB



config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...


  Building C — seed=623, α=0.5...


  [C_finetuned_seed623_alpha0.5]: 100%|██████████| 12612/12612 [02:09<00:00, 97.51it/s] 


  ✅ Indexed: 12612 | Skipped: 0 | Saved: C_finetuned_seed623_alpha0.5

  Building C — seed=623, α=0.7...


  [C_finetuned_seed623_alpha0.7]: 100%|██████████| 12612/12612 [02:01<00:00, 104.11it/s]


  ✅ Indexed: 12612 | Skipped: 0 | Saved: C_finetuned_seed623_alpha0.7

All indexes built!
  A_vision_only_alpha1.0_index.bin  (29.4 MB)
  A_vision_only_alpha1.0_metadata.json  (1.6 MB)
  B_frozen_alpha0.5_index.bin  (29.4 MB)
  B_frozen_alpha0.5_metadata.json  (1.6 MB)
  B_frozen_alpha0.7_index.bin  (29.4 MB)
  B_frozen_alpha0.7_metadata.json  (1.6 MB)
  C_finetuned_seed42_alpha0.5_index.bin  (29.4 MB)
  C_finetuned_seed42_alpha0.5_metadata.json  (1.6 MB)
  C_finetuned_seed42_alpha0.7_index.bin  (29.4 MB)
  C_finetuned_seed42_alpha0.7_metadata.json  (1.6 MB)
  C_finetuned_seed601_alpha0.5_index.bin  (29.4 MB)
  C_finetuned_seed601_alpha0.5_metadata.json  (1.6 MB)
  C_finetuned_seed601_alpha0.7_index.bin  (29.4 MB)
  C_finetuned_seed601_alpha0.7_metadata.json  (1.6 MB)
  C_finetuned_seed606_alpha0.5_index.bin  (29.4 MB)
  C_finetuned_seed606_alpha0.5_metadata.json  (1.6 MB)
  C_finetuned_seed606_alpha0.7_index.bin  (29.4 MB)
  C_finetuned_seed606_alpha0.7_metadata.json  (1.6 MB)
  C_fin